In [1]:
import pandas as pd

df = pd.read_csv(
    "outputs/deduplicated_papers.csv"
)

df.head()

,query,source,source_id,title,authors,abstract,keywords,year,doi,url,doi_normalized,title_normalized
0,machine learning healthcare,PubMed,42876421,Bionic Structural Design of Multifunctional Fl...,"['Mingqian Sheng', 'Mingyang Qiu', 'Enchong Ni...",Rehabilitation medicine is an essential compon...,"['CKD', 'electronic skin', 'machine learning',...",2026.0,10.1002/smtd.71134,https://pubmed.ncbi.nlm.nih.gov/42876421/,10.1002/smtd.71134,bionic structural design of multifunctional fl...
1,machine learning healthcare,PubMed,42876358,Clinical Decision Support Models to Optimize C...,"['Robin Z Y Lu', 'Toni J F Michael', 'Vinayak ...",Controlled ovarian hyperstimulation (COH) is a...,"['clinical decision support', 'controlled ovar...",2026.0,10.1111/cts.70742,https://pubmed.ncbi.nlm.nih.gov/42876358/,10.1111/cts.70742,clinical decision support models to optimize c...
2,machine learning healthcare,PubMed,42876042,Machine learning predictors of post-radiothera...,"['Víctor Aguirre-Bravo', 'Rodrigo El Far', 'Ma...",BACKGROUND: Most evidence on post-radiotherapy...,"['AHA PREVENT score', 'Firth logistic regressi...",2026.0,10.1002/acm2.70862,https://pubmed.ncbi.nlm.nih.gov/42876042/,10.1002/acm2.70862,machine learning predictors of postradiotherap...
3,machine learning healthcare,PubMed,42875776,Mismatch between urban eco-sanitation improvem...,"['Zhen Peng', 'Lin Wang']",BACKGROUND: Health service supply capacity dep...,"['Chinese cities', 'Health service governance'...",2026.0,10.1186/s12913-026-15444-8,https://pubmed.ncbi.nlm.nih.gov/42875776/,10.1186/s12913-026-15444-8,mismatch between urban ecosanitation improveme...
4,machine learning healthcare,PubMed,42875668,Predicting Hospital-associated complications i...,"['Shukhrat Valiev', 'Dilnavoz Yuldasheva', 'Ma...",Machine learning is increasingly used to predi...,"['Data leakage', 'Machine learning', 'Post-sur...",2026.0,10.1007/s40520-026-03507-z,https://pubmed.ncbi.nlm.nih.gov/42875668/,10.1007/s40520-026-03507-z,predicting hospitalassociated complications in...


In [2]:
df[
    ["title", "abstract", "keywords"]
].isna().sum()

title         1
abstract    111
keywords      0
dtype: int64

In [3]:
def build_embedding_text(row):
    parts = []

    if pd.notna(row["title"]):
        parts.append(str(row["title"]))

    if pd.notna(row["abstract"]):
        parts.append(str(row["abstract"]))

    if pd.notna(row["keywords"]):
        parts.append(str(row["keywords"]))

    return " ".join(parts)

In [4]:
df["embedding_text"] = df.apply(
    build_embedding_text,
    axis=1
)

In [5]:
df[
    ["title", "abstract", "keywords", "embedding_text"]
].head()

,title,abstract,keywords,embedding_text
0,Bionic Structural Design of Multifunctional Fl...,Rehabilitation medicine is an essential compon...,"['CKD', 'electronic skin', 'machine learning',...",Bionic Structural Design of Multifunctional Fl...
1,Clinical Decision Support Models to Optimize C...,Controlled ovarian hyperstimulation (COH) is a...,"['clinical decision support', 'controlled ovar...",Clinical Decision Support Models to Optimize C...
2,Machine learning predictors of post-radiothera...,BACKGROUND: Most evidence on post-radiotherapy...,"['AHA PREVENT score', 'Firth logistic regressi...",Machine learning predictors of post-radiothera...
3,Mismatch between urban eco-sanitation improvem...,BACKGROUND: Health service supply capacity dep...,"['Chinese cities', 'Health service governance'...",Mismatch between urban eco-sanitation improvem...
4,Predicting Hospital-associated complications i...,Machine learning is increasingly used to predi...,"['Data leakage', 'Machine learning', 'Post-sur...",Predicting Hospital-associated complications i...


In [6]:
(df["embedding_text"].str.strip() == "").sum()

np.int64(0)

In [7]:
df["has_abstract"] = df["abstract"].notna()

In [8]:
from sentence_transformers import SentenceTransformer

W1020 15:41:40.468000 78338 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [9]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
paper_embeddings = model.encode(
    df["embedding_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

In [11]:
paper_embeddings.shape

(339, 384)

In [12]:
query = "climate change agriculture"

query_embedding = model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_embedding.shape

(384,)

In [13]:
import numpy as np

similarity_scores = paper_embeddings @ query_embedding

In [14]:
df["similarity_score"] = similarity_scores

In [15]:
ranked_df = df.sort_values(
    "similarity_score",
    ascending=False
).reset_index(drop=True)

In [18]:
ranked_df[
    [
        "query",
        "source",
        "title",
        "year",
        "doi",
        "similarity_score",
        "has_abstract"
    ]
].head(10)

,query,source,title,year,doi,similarity_score,has_abstract
0,climate change agriculture,Crossref,Climate Change and Agriculture,2023.0,10.1093/hesc/9780198807506.003.0005,0.825240,True
1,climate change agriculture,Crossref,Climate-Smart Agriculture and International Cl...,2016.0,10.1007/978-94-017-7462-8_25,0.762563,False
2,climate change agriculture,Crossref,Resiliency of Prairie Agriculture to Climate C...,2019.0,10.5772/intechopen.87098,0.746141,False
3,food security climate change,Crossref,Climate change and food security,2026.0,10.1016/b978-0-443-41741-2.00075-2,0.646265,False
4,food security climate change,Crossref,Agronomy-Food Security-Climate Change and the ...,2020.0,10.5772/intechopen.92690,0.638198,False
5,food security climate change,Crossref,Food Security and Climate Change 101,NaN,10.1163/9789004322714_cclc_2022-0216-0801,0.613191,False
6,food security climate change,Crossref,Climate Change and Food Security in India,2010.0,10.1007/978-90-481-9516-9_15,0.601802,False
7,food security climate change,PubMed,Emerging livestock science and technologies ca...,2026.0,10.1007/s43621-026-04101-z,0.591838,True
8,climate change agriculture,Crossref,Agriculture,NaN,10.1017/cbo9780511546013.012,0.557882,False
9,remote sensing agriculture,Crossref,REMOTE SENSING IN AGRICULTURE: PROGRESS AND PR...,1990.0,10.1016/b978-0-408-04767-8.50028-1,0.553163,False


In [17]:
print("Highest:", ranked_df["similarity_score"].max())
print("Lowest:", ranked_df["similarity_score"].min())
print("Mean:", ranked_df["similarity_score"].mean())

Highest: 0.82524014
Lowest: -0.13408753
Mean: 0.08722672


In [19]:
query_score_summary = (
    ranked_df
    .groupby("query")["similarity_score"]
    .agg(["mean", "max", "count"])
    .sort_values("mean", ascending=False)
)

query_score_summary

,mean,max,count
query,,,
food security climate change,0.476505,0.646265,11
climate change agriculture,0.421058,0.825240,14
remote sensing agriculture,0.324304,0.553163,14
biodiversity conservation,0.212576,0.272356,15
air pollution human health,0.209423,0.303285,10
solar energy forecasting,0.195549,0.355247,15
urban sustainability,0.169809,0.289028,15
economic inequality,0.103505,0.355644,15
renewable energy storage,0.076214,0.193820,15


In [20]:
ranked_df.groupby("has_abstract")["similarity_score"].agg(
    ["mean", "median", "count"]
)

,mean,median,count
has_abstract,,,
False,0.119456,0.049242,111
True,0.071536,0.021863,228


In [21]:
ranked_df[
    ["source", "similarity_score"]
].groupby("source").agg(
    ["mean", "median", "count"]
)

similarity_score                
                     mean    median count
source                                   
Crossref         0.127003  0.049242   125
PubMed           0.061246  0.022926   124
arXiv            0.067778  0.023218    90

In [22]:
ranked_df.groupby(
    ["source", "has_abstract"]
)["similarity_score"].agg(
    ["mean", "median", "count"]
)

mean    median  count
source   has_abstract                           
Crossref False         0.122180  0.057346    108
         True          0.157644  0.016250     17
PubMed   False         0.021399  0.024990      3
         True          0.062234  0.022391    121
arXiv    True          0.067778  0.023218     90

In [23]:
for source in ranked_df["source"].unique():
    print(f"\n--- {source} ---")

    display(
        ranked_df[
            ranked_df["source"] == source
        ][
            ["title", "similarity_score", "has_abstract"]
        ].head(5)
    )


--- Crossref ---


,title,similarity_score,has_abstract
0,Climate Change and Agriculture,0.825240,True
1,Climate-Smart Agriculture and International Cl...,0.762563,False
2,Resiliency of Prairie Agriculture to Climate C...,0.746141,False
3,Climate change and food security,0.646265,False
4,Agronomy-Food Security-Climate Change and the ...,0.638198,False



--- PubMed ---


,title,similarity_score,has_abstract
7,Emerging livestock science and technologies ca...,0.591838,True
18,UAV-Based High-Throughput Phenotyping for Earl...,0.425197,True
19,Advances in plant metabolic engineering throug...,0.371491,True
20,Global inequalities in weather forecasts.,0.355644,True
21,Mapping and dynamic monitoring of desertificat...,0.355247,True



--- arXiv ---


,title,similarity_score,has_abstract
16,Intelligent Agricultural Greenhouse Control Sy...,0.490622,True
17,Agricultural Landscape Understanding At Countr...,0.466349,True
25,Jalisco's multiclass land cover analysis and c...,0.307344,True
41,A Temporally Consistent Image-based Sun Tracki...,0.261262,True
43,Improving Model Chain Approaches for Probabili...,0.257706,True


In [24]:
for source in ranked_df["source"].unique():

    print(f"\n--- {source} ---")

    display(
        ranked_df[
            ranked_df["source"] == source
        ][
            [
                "query",
                "title",
                "similarity_score",
                "has_abstract"
            ]
        ].head(10)
    )


--- Crossref ---


,query,title,similarity_score,has_abstract
0,climate change agriculture,Climate Change and Agriculture,0.825240,True
1,climate change agriculture,Climate-Smart Agriculture and International Cl...,0.762563,False
2,climate change agriculture,Resiliency of Prairie Agriculture to Climate C...,0.746141,False
3,food security climate change,Climate change and food security,0.646265,False
4,food security climate change,Agronomy-Food Security-Climate Change and the ...,0.638198,False
5,food security climate change,Food Security and Climate Change 101,0.613191,False
6,food security climate change,Climate Change and Food Security in India,0.601802,False
8,climate change agriculture,Agriculture,0.557882,False
9,remote sensing agriculture,REMOTE SENSING IN AGRICULTURE: PROGRESS AND PR...,0.553163,False
10,climate change agriculture,"Climate Change, Agriculture, and Famine",0.543977,True



--- PubMed ---


,query,title,similarity_score,has_abstract
7,food security climate change,Emerging livestock science and technologies ca...,0.591838,True
18,remote sensing agriculture,UAV-Based High-Throughput Phenotyping for Earl...,0.425197,True
19,food security climate change,Advances in plant metabolic engineering throug...,0.371491,True
20,economic inequality,Global inequalities in weather forecasts.,0.355644,True
21,solar energy forecasting,Mapping and dynamic monitoring of desertificat...,0.355247,True
22,food security climate change,CRISPR genome editing in food and feed crops: ...,0.336819,True
23,climate change agriculture,Evolutionary constraint on a core winter prote...,0.321504,True
24,remote sensing agriculture,The performance of directional area scattering...,0.316895,True
28,climate change agriculture,Regulation of antioxidant defense and stress p...,0.298045,True
31,solar energy forecasting,Predicting global horizontal irradiance using ...,0.284041,True



--- arXiv ---


,query,title,similarity_score,has_abstract
16,climate change agriculture,Intelligent Agricultural Greenhouse Control Sy...,0.490622,True
17,food security climate change,Agricultural Landscape Understanding At Countr...,0.466349,True
25,climate change agriculture,Jalisco's multiclass land cover analysis and c...,0.307344,True
41,solar energy forecasting,A Temporally Consistent Image-based Sun Tracki...,0.261262,True
43,solar energy forecasting,Improving Model Chain Approaches for Probabili...,0.257706,True
44,urban sustainability,How are cities pledging net zero? A computatio...,0.256359,True
47,solar energy forecasting,Weather Forecasting Error in Solar Energy Fore...,0.248833,True
48,climate change agriculture,Automatic Counting and Classification of Mosqu...,0.247175,True
52,biodiversity conservation,Preserving Nature's Ledger: Blockchains in Bio...,0.239340,True
54,climate change agriculture,How Does the Spatial Distribution of Pre-train...,0.235523,True


In [25]:
ranked_df.groupby("source")[
    "similarity_score"
].describe()

,count,mean,std,min,25%,50%,75%,max
source,,,,,,,,
Crossref,125.0,0.127003,0.207536,-0.104480,-0.011900,0.049242,0.211945,0.825240
PubMed,124.0,0.061246,0.133215,-0.134088,-0.032649,0.022926,0.129017,0.591838
arXiv,90.0,0.067778,0.114920,-0.115991,-0.014918,0.023218,0.140439,0.490622
